In [ ]:
import pandas as pd
from modules.load_config import load_params_from_yaml

import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, tf_to_int, tf_to_daily, tf_to_hourly
from modules.visualisations import plot_profile_by_category,plot_distribution_comparison,plot_sorted_mps_comparison

from plotly.io import to_html
from IPython.display import display, HTML

from modules.optimization_algorithms.EqualWaterfillingOptAlgo import EqualWaterfillingOptAlgo
from modules.optimization_algorithms.NashProductOptAlgo import NashProductOptAlgo
from modules.optimization_algorithms.OptAlgo import OptAlgo
from datetime import datetime, UTC

from pathlib import Path
import json


In [ ]:
# 
# LOAD ENPARTO LOGO
# 
image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

- from the perspective of a GEN MP: 
    - how much is produced
    - how much is distributed in the REC
    - how much is now distributed in the REC
    - how much is now distributed in X
    - how much is distributed IN TOTAL  / change of total surplus

- from the perspective of a CONS MP: 
    - how much is consumed
    - how much is covered by the REC
    - how much is now covered by the REC
    - how much is now covered by the REC
    - how much is covered IN TOTAL  / change of total Comm Cov

- from the perspective of a single REC:
    - sums of all energy flow values
    - sums of all energy flow INTERNAL values
    - sums of all energy flow to X values

## Params

In [ ]:
# # PARMS
# changeable
consumer_org_ids = [1] # [1, 2]
prosumer_org_ids = [13] # [13, 17]

start_time = datetime(2025, 2, 20)
end_time = datetime(2025, 2, 23)

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

## load data

In [ ]:
config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename_template = config_dict["SINGLE_MPS_OF_EEG"]

In [ ]:
raw_eeg = []

for eeg_type, eeg_ids_of_type in [
    ("c", consumer_org_ids),
    ("p", prosumer_org_ids),
]:
    for act_org_id in eeg_ids_of_type:
        act_filepath_to_load = (
            f"{path_to_local_data}{input_filename_template.format(org_id=act_org_id)}"
        )
        act_df = pd.read_csv(act_filepath_to_load)
        act_df["eeg_type"] = eeg_type
        raw_eeg.append(act_df)

# row-wise append into a single DataFrame
raw_eeg_df = pd.concat(raw_eeg, ignore_index=True)

raw_eeg_df['time'] = pd.to_datetime(raw_eeg_df['time'], utc=True)

eeg_selected_time_horizon = raw_eeg_df[
    (raw_eeg_df["time"] > pd.Timestamp(start_time, tz='UTC')) &
    (raw_eeg_df["time"] < pd.Timestamp(end_time, tz='UTC'))
]
del raw_eeg_df



print(f"{eeg_selected_time_horizon.dtypes}")
print(f"len: {len(eeg_selected_time_horizon)}")
eeg_selected_time_horizon.head()

In [ ]:
eeg_selected_feat = eeg_selected_time_horizon[["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_pot", "comm_cov", "wt_meas_gen", "wt_surp_gen"]].copy()
del eeg_selected_time_horizon

# cons_gen: Consumed Generation, how much of the generated electricity was consumed within the EEG
eeg_selected_feat.sum(numeric_only=True)

# PRINT DF INFO
# 
print(eeg_selected_feat.dtypes)
print(f"len: {len(eeg_selected_feat)}")
print(eeg_selected_feat.drop(["organization_id", "metering_point_id"], axis=1).describe())
eeg_selected_feat.head()

#
# FEATURE ENGINEERIN
#

eeg_selected_feat["cons_gen"] = eeg_selected_feat["wt_meas_gen"] - eeg_selected_feat["wt_surp_gen"]

In [ ]:
#
# DETECT NON-PV GENERATION
# 

# mp_id, where energy_direction = "G", and q75 of wt_meas_gen between 11 and 3 is greater than 1.

In [ ]:
eeg_selected_feat[eeg_selected_feat["organization_id"]==13].groupby("time").sum(numeric_only=True)

In [ ]:
work = eeg_selected_feat.drop_duplicates().copy()
del eeg_selected_feat

org_algo_mapping = {
    13: {"opt_algo": EqualWaterfillingOptAlgo, "share_ratio": 1},
    1: {"opt_algo": EqualWaterfillingOptAlgo, "share_ratio": 1},
    "x": {
        "opt_algo": EqualWaterfillingOptAlgo,
        "share_ratio": 1,
    },
}

# result container (separate from config!)
simulated_energy_data = {}

run_timestamp = datetime.now(UTC).isoformat()

for act_org_id in [13, 1]:  # work["organization_id"].unique():
    print(f"start opt {act_org_id}:")

    algo_cls = org_algo_mapping[act_org_id]["opt_algo"]
    act_algo: OptAlgo = algo_cls()

    act_work = work[work["organization_id"] == act_org_id].copy()

    act_tf_schedule = act_algo.calculate_pfs(act_work)
    act_sim_ed = apply_pf_schedule_to_mps(act_work, act_tf_schedule)

    simulated_energy_data[act_org_id] = {
        "tf_schedule": act_tf_schedule,
        "simulated_energy_data": act_sim_ed,
        "algo_name": algo_cls.__name__,
        "algo_params": vars(act_algo),  # assumes params stored on self
        "created_at": run_timestamp,
    }
    print(f"end opt {act_org_id}:")

In [ ]:
temp_work = pd.concat([simulated_energy_data[1]["simulated_energy_data"], simulated_energy_data[13]["simulated_energy_data"]]).copy()

In [ ]:
temp_work["rest_wt_meas_gen"] = temp_work["wt_meas_gen"] * (1 - temp_work["pf"]) 
temp_work["rest_wt_meas_cons"] = temp_work["wt_meas_cons"] * (1 - temp_work["pf"]) 

# extract org_id -> share_ratio

share_ratio_map = {org_id: cfg["share_ratio"] for org_id, cfg in org_algo_mapping.items()}
temp_work["transfer_ratio"] = temp_work["organization_id"].map(share_ratio_map)
temp_work["transfer_wt_meas_cons"] = temp_work["rest_wt_meas_cons"] * temp_work["transfer_ratio"]
temp_work["transfer_wt_meas_gen"] = temp_work["rest_wt_meas_gen"] * temp_work["transfer_ratio"]

In [ ]:
temp_work[temp_work["energy_direction"]=="G"][["energy_direction", "wt_meas_gen", "transfer_wt_meas_gen", "pf"]]

In [ ]:
temp_work[temp_work["pf"]!=1][["organization_id", "wt_meas_gen", "opt_wt_meas_gen", "rest_wt_meas_gen", "transfer_wt_meas_gen", "transfer_ratio"]]


sim_eeg_combination = temp_work[["organization_id", "metering_point_id", "energy_direction", "time", "transfer_wt_meas_gen", "transfer_wt_meas_cons"]].rename(columns={"transfer_wt_meas_gen":"wt_meas_gen", "transfer_wt_meas_cons":"wt_meas_cons"})

new_calced_agg_on_time = sim_eeg_combination.groupby(by="time").sum().reset_index()[["time", "wt_meas_cons", "wt_meas_gen"]].rename(columns={"wt_meas_cons":"sum_wt_meas_cons", "wt_meas_gen":"sum_wt_meas_gen"})
sim_eeg_combination = sim_eeg_combination.merge(new_calced_agg_on_time, on=["time"], how="left")


sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"] = (sim_eeg_combination["sum_wt_meas_gen"] / sim_eeg_combination["sum_wt_meas_cons"]).replace(np.nan, 1)
sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio_clipped"] = sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"].clip(upper=1) 
sim_eeg_combination["wt_meas_surp_to_wt_meas_gen_ratio_clipped"] = ((sim_eeg_combination["sum_wt_meas_gen"] - sim_eeg_combination["sum_wt_meas_cons"])/sim_eeg_combination["sum_wt_meas_gen"]).clip(lower=0) 

# applying new rations to calculate new comm_cov, comm_pot and wt_meas_surp after pf schedule application
sim_eeg_combination["comm_cov"] = sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio_clipped"] * sim_eeg_combination["wt_meas_cons"]
sim_eeg_combination["comm_pot"] = sim_eeg_combination["wt_meas_gen_to_wt_meas_cons_ratio"] * sim_eeg_combination["wt_meas_cons"]
sim_eeg_combination["wt_surp_gen"] = sim_eeg_combination["wt_meas_surp_to_wt_meas_gen_ratio_clipped"] * sim_eeg_combination["wt_meas_gen"]
sim_eeg_combination["cons_gen"] = sim_eeg_combination["wt_meas_gen"] - sim_eeg_combination["wt_surp_gen"]

sim_eeg_combination["source_organization_id"]=sim_eeg_combination["organization_id"]
sim_eeg_combination["organization_id"]="x"

sim_eeg_combination.head()

In [ ]:
act_org_id="x"
print(f"start opt {act_org_id}:")

algo_cls = org_algo_mapping[act_org_id]["opt_algo"]
act_algo = algo_cls()

act_work = sim_eeg_combination[sim_eeg_combination["organization_id"] == act_org_id].drop(columns=["sum_wt_meas_cons", "sum_wt_meas_gen"]).copy()

act_tf_schedule = act_algo.calculate_pfs(act_work)
act_sim_ed = apply_pf_schedule_to_mps(act_work, act_tf_schedule)

simulated_energy_data[act_org_id] = {
    "tf_schedule": act_tf_schedule,
    "simulated_energy_data": act_sim_ed,
    "algo_name": algo_cls.__name__,
    "algo_params": vars(act_algo),  # assumes params stored on self
    "created_at": run_timestamp,
}
print(f"end opt {act_org_id}:")

In [ ]:
def print_opt_energy_data(applied_pfs: pd.DataFrame) -> None:
    """
    Output of optimized values after participation factor schedule apply.
    Dynamically prints all relevant energy features with sums and comparisons.
    """

    # baseline & optimized Restnetzbezug
    baseline_restnetzbezug = (
        applied_pfs["wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum()
    )
    opt_restnetzbezug = (
        applied_pfs["opt_wt_meas_cons"].sum() - applied_pfs["comm_cov"].sum()
    )

    print("\nOptimized Sums:")

    # Features to report: dynamically handle all columns starting with 'opt_'
    cons_feature_map = {
        "opt_wt_meas_cons": ("opt_meas_cons (c*)", "wt_meas_cons"),
        "opt_comm_cov": ("opt_comm_cov (cc*)", "comm_cov"),
        "opt_comm_pot": ("opt_comm_pot (cp*)", "comm_pot"),
    }

    gen_feature_map = {
        "opt_wt_meas_gen": ("opt_wt_meas_gen (g*)", "wt_meas_gen"),
        "opt_wt_surp_gen": ("opt_wt_surp_gen (s*)", "wt_surp_gen"),
        "opt_cons_gen": ("opt_cons_gen (cg*)", "cons_gen"),
    }

    def _print_feature_changes(feature_map:dict, category:str) -> None:
        """Prints the changes from baseline energy flow features to optimized energy flow features."""
        print(f"\t{category}")
        for opt_col, (label, base_col) in feature_map.items():
            opt_sum = applied_pfs[opt_col].sum()
            if base_col:
                base_sum = applied_pfs[base_col].sum()
                diff = base_sum - opt_sum
                print(
                    f"\t\t{label}: {opt_sum:.3f} [base {base_col}: {base_sum:.3f}, difference: {diff:.3f}]"
                )
            else:
                print(f"\t\t{label}: {opt_sum:.3f}")

    _print_feature_changes(cons_feature_map, "Consumers:")
    

    print(f"\t-> opt_Restnetzbezug (c* - cc*): {opt_restnetzbezug:.3f}")
    print(f"\t-> real Restnetzbezug: {baseline_restnetzbezug:.3f}")
    print(
        f"\t-> real Restnetzbezug + opt_comm_cov = {baseline_restnetzbezug:.3f} + {applied_pfs['opt_comm_cov'].sum():.3f} = {applied_pfs['wt_meas_cons'].sum():.3f}"
    )

    _print_feature_changes(gen_feature_map, "Generators:")

In [ ]:
source_eeg_suffix = "_s"
combined_eeg_suffix = "_c"

comb_eegs = simulated_energy_data[1]["simulated_energy_data"].merge(
    simulated_energy_data["x"]["simulated_energy_data"],
    on=["time", "metering_point_id", "energy_direction"],
    suffixes=(source_eeg_suffix, combined_eeg_suffix),
)

comb_eegs[f"pf{source_eeg_suffix}_to{combined_eeg_suffix}"] = comb_eegs[f"pf{combined_eeg_suffix}"]
comb_eegs[f"pf{combined_eeg_suffix}"] = comb_eegs[f"pf{source_eeg_suffix}_to{combined_eeg_suffix}"] * comb_eegs[f"pf{source_eeg_suffix}"]

comb_eegs[comb_eegs["energy_direction"] == "G"][
    [
        "time",
        "metering_point_id",
        "energy_direction",
        f"wt_meas_gen{source_eeg_suffix}",
        f"opt_wt_meas_gen{source_eeg_suffix}",
        f"wt_meas_gen{combined_eeg_suffix}",
        f"opt_wt_meas_gen{combined_eeg_suffix}",
        f"pf{source_eeg_suffix}",
        f"pf{combined_eeg_suffix}",
        f"pf{source_eeg_suffix}_to{combined_eeg_suffix}"
    ]
]

# wt_meas_gen_s
# opt_wt_meas_gen_s = wt_meas_gen_s * pf_s
# wt_meas_gen_c = wt_meas_gen_s * (1 - pf_s)
# wt_meas_gen_s = opt_wt_meas_gen_s + wt_meas_gen_c
# TODO

In [ ]:
print_opt_energy_data(simulated_energy_data[1]["simulated_energy_data"])
print_opt_energy_data(simulated_energy_data[13]["simulated_energy_data"])
print_opt_energy_data(simulated_energy_data["x"]["simulated_energy_data"])

In [ ]:
def gini(array: np.ndarray) -> float:
    """Compute Gini coefficient of a numpy array."""
    array = array.flatten()
    if np.all(array == 0):
        return 0.0
    sorted_array = np.sort(array)
    n = len(array)
    cumvals = np.cumsum(sorted_array)
    return (n + 1 - 2 * np.sum(cumvals) / cumvals[-1]) / n



In [ ]:
simulated_energy_data["x"]["simulated_energy_data"][["time", "organization_id", "metering_point_id", "energy_direction", "wt_meas_cons", "comm_cov", "comm_pot", "wt_meas_gen", "wt_surp_gen", "cons_gen", "pf", "opt_wt_meas_cons", "opt_comm_cov", "opt_comm_pot", "opt_wt_meas_gen", "opt_wt_surp_gen", "opt_cons_gen",]]

In [ ]:
move = simulated_energy_data[1]["simulated_energy_data"][(simulated_energy_data[1]["simulated_energy_data"]["energy_direction"]=="G") & ((simulated_energy_data[1]["simulated_energy_data"]["cons_gen"] - simulated_energy_data[1]["simulated_energy_data"]["opt_cons_gen"]).abs() <= 5e-4) & (simulated_energy_data[1]["simulated_energy_data"]["pf"]!=1)]

print(
move[["time", "metering_point_id", "wt_meas_gen", "cons_gen", "opt_wt_meas_gen", "opt_cons_gen", "pf"]].sort_values("wt_meas_gen", ascending=False).head())

In [ ]:
dataframes = []

for org_id, values in simulated_energy_data.items():
    act_sim_ed = values["simulated_energy_data"].copy()
    dataframes.append(act_sim_ed)

all_orgs_ed = pd.concat(dataframes)

In [ ]:
all_orgs_ed["organization_id"].unique()

In [ ]:
all_orgs_ed.columns

In [ ]:
def save_file():
    base_path = Path("../../local_data/")
    base_path.mkdir(exist_ok=True)


    for org_id, result in simulated_energy_data.items():

        org_path = base_path / f"org_{org_id}" / result["algo_name"]
        org_path.mkdir(parents=True, exist_ok=True)

        # 1️⃣ TF schedule
        result["tf_schedule"].to_csv(
            org_path / f"tf_schedule_{run_timestamp}.csv",
            index=False,
        )

        # 2️⃣ Simulated energy data
        result["simulated_energy_data"].to_csv(
            org_path / f"simulated_energy_{run_timestamp}.csv",
            index=False,
        )

        # 3️⃣ Metadata
        metadata = {
            "organization_id": org_id,
            "algo": result["algo_name"],
            "algo_params": result["algo_params"],
            "created_at": result["created_at"],
            "row_counts": {
                "tf_schedule": len(result["tf_schedule"]),
                "simulated_energy": len(result["simulated_energy_data"]),
            },
        }

        with open(org_path / f"metadata_{run_timestamp}.json", "w") as f:
            json.dump(metadata, f, indent=2)


    

In [ ]:
# TODO 
# 6. expand in the report with images -> red/green chart
# Multi REC text 7. in the report
# # When, purpose, scenario, charts, ...
# # 

# check duplicates -> where do they come from?
# # L1/L2, data quality???

# 2 RECs

# 3 scenarios
- both surplus: do nothing
- both under-coverage: do nothing
- 1 surplus, 1 under-coverage